In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (log_loss, roc_auc_score, f1_score, 
                            accuracy_score, precision_score, recall_score, matthews_corrcoef)
import optuna 

In [ ]:
X_train = pd.read_csv('../datasets/preprocessed/X_train2.csv')
y_train = pd.read_csv('../datasets/preprocessed/y_train2.csv').squeeze()  # Series 변환
X_val = pd.read_csv('../datasets/preprocessed/X_val2.csv')
y_val = pd.read_csv('../datasets/preprocessed/y_val2.csv').squeeze()

In [ ]:
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise']),
        'eval_metric': 'Logloss',
        'early_stopping_rounds': 50,
        'verbose': False,
        'random_seed': 42
    }
    
    model = CatBoostClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=list(X_train.select_dtypes(include='object').columns),
        use_best_model=True
    )
    preds_proba = model.predict_proba(X_val)[:, 1]
    return log_loss(y_val, preds_proba)

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  

[I 2025-05-29 02:07:16,232] A new study created in memory with name: no-name-21b3a5a0-da13-47fb-8758-321eb49310e1
[I 2025-05-29 02:07:20,729] Trial 0 finished with value: 0.6177247614758733 and parameters: {'learning_rate': 0.014279714304742496, 'depth': 10, 'l2_leaf_reg': 0.9551469764444749, 'random_strength': 0.19588846631710882, 'border_count': 87, 'grow_policy': 'SymmetricTree'}. Best is trial 0 with value: 0.6177247614758733.
[I 2025-05-29 02:07:26,536] Trial 1 finished with value: 0.6224938322320162 and parameters: {'learning_rate': 0.011032132746429934, 'depth': 9, 'l2_leaf_reg': 0.0036565499340118664, 'random_strength': 1.5299366203690583, 'border_count': 65, 'grow_policy': 'Depthwise'}. Best is trial 0 with value: 0.6177247614758733.
[I 2025-05-29 02:07:27,191] Trial 2 finished with value: 0.6256865365981439 and parameters: {'learning_rate': 0.09237155042892174, 'depth': 8, 'l2_leaf_reg': 0.17775138448973155, 'random_strength': 4.039181665173, 'border_count': 93, 'grow_policy'

In [ ]:
best_params = study.best_params
best_params.update({
    'eval_metric': 'Logloss',
    'early_stopping_rounds': 50,
    'random_seed': 42
})

final_model = CatBoostClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=list(X_train.select_dtypes(include='object').columns),
    use_best_model=True
)

0:	learn: 0.6908836	test: 0.6908427	best: 0.6908427 (0)	total: 8.42ms	remaining: 8.41s
1:	learn: 0.6890993	test: 0.6890781	best: 0.6890781 (1)	total: 17.1ms	remaining: 8.54s
2:	learn: 0.6877077	test: 0.6875911	best: 0.6875911 (2)	total: 21.1ms	remaining: 7.01s
3:	learn: 0.6865505	test: 0.6863192	best: 0.6863192 (3)	total: 24.9ms	remaining: 6.21s
4:	learn: 0.6854799	test: 0.6852839	best: 0.6852839 (4)	total: 28.6ms	remaining: 5.7s
5:	learn: 0.6839537	test: 0.6838247	best: 0.6838247 (5)	total: 32.6ms	remaining: 5.39s
6:	learn: 0.6826515	test: 0.6826344	best: 0.6826344 (6)	total: 36.3ms	remaining: 5.14s
7:	learn: 0.6814013	test: 0.6814235	best: 0.6814235 (7)	total: 40.3ms	remaining: 5s
8:	learn: 0.6804499	test: 0.6805754	best: 0.6805754 (8)	total: 44.1ms	remaining: 4.86s
9:	learn: 0.6791906	test: 0.6793569	best: 0.6793569 (9)	total: 48.1ms	remaining: 4.76s
10:	learn: 0.6779428	test: 0.6782706	best: 0.6782706 (10)	total: 52.2ms	remaining: 4.69s
11:	learn: 0.6766260	test: 0.6771059	best: 0.

In [ ]:
from sklearn.metrics import f1_score

def find_optimal_threshold(model, X_val, y_val):
    y_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 50)
    best_thresh = 0.5
    best_f1 = 0
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        f1 = f1_score(y_val, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1

optimal_threshold, best_f1 = find_optimal_threshold(final_model, X_val, y_val)
print(f"Optimal Threshold: {optimal_threshold:.4f}, Best F1: {best_f1:.4f}")


Optimal Threshold: 0.3449, Best F1: 0.6967


In [ ]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    return {
        'logloss': log_loss(y, y_proba),
        'auc': roc_auc_score(y, y_proba),
        'f1': f1_score(y, y_pred),
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred),
        'recall': recall_score(y, y_pred),
        'mcc': matthews_corrcoef(y, y_pred)
    }

In [ ]:
val_metrics = evaluate_model(final_model, X_val, y_val, optimal_threshold)

In [ ]:
print("\n" + "="*50)
print(f"{'Metric':<12} {'Value':<10}")
print("-"*50)
for metric, value in val_metrics.items():
    print(f"{metric:<12} {value:.6f}")
print("="*50)


Metric       Value     
--------------------------------------------------
logloss      0.614193
auc          0.720814
f1           0.696653
accuracy     0.614015
precision    0.570641
recall       0.894091
mcc          0.279474


In [ ]:
feature_importances = final_model.get_feature_importance()
feature_names = X_train.columns
sorted_idx = np.argsort(feature_importances)[::-1]
print("\nFeature Importances:")
for i in range(min(10, len(feature_names))):
    print(f"{i+1}. {feature_names[sorted_idx[i]]}: {feature_importances[sorted_idx[i]]:.4f}")


Feature Importances:
1. kw_avg_avg: 7.9363
2. kw_max_avg: 6.2398
3. data_channel: 6.1941
4. weekday: 6.0744
5. self_reference_min_shares: 5.0671
6. kw_min_avg: 4.4601
7. LDA_04: 4.1345
8. self_reference_avg_sharess: 3.4987
9. LDA_00: 3.1425
10. LDA_01: 3.0292


In [ ]:
from sklearn.metrics import confusion_matrix
y_pred_val = (final_model.predict_proba(X_val)[:, 1] >= optimal_threshold).astype(int)
cm = confusion_matrix(y_val, y_pred_val)
print("\nConfusion Matrix")
print(pd.DataFrame(cm, 
                   index=['Actual 0', 'Actual 1'],
                   columns=['Predicted 0', 'Predicted 1']))


Confusion Matrix:
          Predicted 0  Predicted 1
Actual 0          758         1480
Actual 1          233         1967
